# Week 2-2 Skeleton: Transform 심화 - 비즈니스 테이블 만들고 분석하기

이번 실습은 지난 시간에 만든 `movies_clean` 중간 테이블에서 출발합니다.

목표는 원본 데이터를 한 번 더 다듬고 조인해서 분석에 재사용할 수 있는 중간 테이블을 만들고, 그 중간 테이블을 바탕으로 사람이 바로 읽거나 BI/시각화에서 사용할 수 있는 비즈니스 테이블을 만드는 것입니다.

이 버전은 수업 중 직접 채워볼 핵심 코드만 일부 비워둔 스켈레톤입니다.

## 0. 이번 시간에 풀 비즈니스 질문

먼저 질문을 정하고 시작합니다. 그래야 왜 `keywords`를 붙이는지, 왜 `profit`, `roi` 같은 파생 컬럼을 만드는지 방향이 분명해집니다.

이번 실습에서 답할 질문은 다음과 같습니다.

1. **신뢰할 만한 고평점 영화는 어느 시기에 많았을까?**
   - 단순히 평점 평균만 보지 않고 `vote_count`도 함께 고려합니다.
2. **대표 키워드별로 수익성과 효율은 어떻게 다를까?**
   - 그래서 `keywords.csv`에서 `main_keyword`를 추출해 영화 정보에 붙입니다.
   - 그래서 `profit`, `roi` 파생 컬럼을 만듭니다.
3. **시기별로 어떤 대표 키워드가 많이 등장했을까?**
   - 연도 단위는 너무 듬성듬성할 수 있어서, 이번에는 `release_decade`를 사용합니다.
4. **wide table과 long table은 각각 언제 쓰기 좋을까?**
   - `pivot_table()`과 `melt()`를 비교합니다.

## 1. Transform의 3단계

이번 실습의 전체 흐름은 아래와 같습니다.

```text
raw
→ intermediate
→ business table
```

- raw: 아직 손대지 않은 원본 데이터
- intermediate: 정제, 조인, 파생 컬럼 생성이 끝난 재사용 가능한 중간 테이블
- business table: 특정 질문에 답하기 위해 집계/피벗된 최종 결과 테이블

## 2. 준비

In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

sns.set_theme(style="whitegrid")

한글 시각화가 깨지지 않도록 사용 가능한 한글 폰트를 찾아 설정합니다.

Mac에서는 보통 `AppleGothic`, Windows에서는 `Malgun Gothic`, 일부 환경에서는 `NanumGothic`을 사용합니다.

In [ ]:
font_candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic"]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}

for font_name in font_candidates:
    if font_name in available_fonts:
        rcParams["font.family"] = font_name
        break

rcParams["axes.unicode_minus"] = False

print("현재 matplotlib 폰트:", rcParams["font.family"])

데이터 경로를 설정합니다. 노트북을 루트에서 실행해도, `week2/src`에서 실행해도 동작하도록 후보 경로를 둡니다.

In [ ]:
DATA_DIR_CANDIDATES = [Path("../data"), Path("week2/data"), Path("data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. download_dataset.py를 먼저 실행하세요.")

RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MOVIES_CLEAN_PATH = OUTPUT_DIR / "movies_clean_for_load.csv"
KEYWORDS_PATH = RAW_DIR / "keywords.csv"

MOVIES_WITH_KEYWORD_PATH = OUTPUT_DIR / "movies_with_keyword.csv"
YEAR_BUSINESS_PATH = OUTPUT_DIR / "year_business_table.csv"
KEYWORD_BUSINESS_WIDE_PATH = OUTPUT_DIR / "keyword_business_table_wide.csv"
KEYWORD_BUSINESS_LONG_PATH = OUTPUT_DIR / "keyword_business_table_long.csv"
DECADE_KEYWORD_PIVOT_PATH = OUTPUT_DIR / "decade_keyword_pivot.csv"

DATA_DIR

출력 숫자가 과학적 표기법(`1.367475e+07`)으로 보이면 처음 보는 입장에서 읽기 어렵습니다.

실습에서는 숫자를 조금 더 읽기 쉽게 보이도록 pandas 출력 포맷을 설정합니다.

In [ ]:
pd.options.display.float_format = "{:,.2f}".format

## 3. 중간 테이블 읽기

지난 실습에서 만든 `movies_clean_for_load.csv`를 읽습니다.

이 파일은 이미 컬럼 선별, 자료형 변환, 결측치/중복/이상치 점검을 거친 중간 정제 테이블입니다.

In [ ]:
movies_clean = pd.read_csv(MOVIES_CLEAN_PATH, parse_dates=["release_date"])

print("movies_clean shape:", movies_clean.shape)
movies_clean.head(3)

In [ ]:
movies_clean.info()

이번 실습에서 사용할 주요 컬럼만 먼저 확인합니다.

In [ ]:
movies_clean[[
    "movie_id", "title", "release_year", "vote_average", "vote_count", "budget", "revenue"
]].head(5)

## 4. keywords 원본 확인

`keywords.csv`는 영화별 키워드 정보를 담고 있습니다.

주의할 점은 `keywords` 컬럼이 일반 문자열처럼 보이지만, 실제로는 리스트 형태의 값이 문자열로 저장되어 있다는 것입니다.

In [ ]:
keywords_raw = pd.read_csv(KEYWORDS_PATH)

print("keywords_raw shape:", keywords_raw.shape)
keywords_raw.head(5)

### 참고: explode는 무엇인가?

`explode`는 한 행 안에 여러 값이 들어 있을 때, 그 여러 값을 여러 행으로 펼치는 작업입니다.

예를 들어 한 영화에 키워드가 3개 있으면 아래처럼 바꿀 수 있습니다.

```text
movie_id | keywords
862      | [toy, friendship, rivalry]
```

```text
movie_id | keyword
862      | toy
862      | friendship
862      | rivalry
```

이 방식은 정석적인 관계 테이블을 만들 때 좋지만, 이번 실습에서는 난이도를 낮추기 위해 전체 explode 대신 첫 번째 키워드 하나만 `main_keyword`로 사용합니다.

## 5. 첫 번째 키워드 추출 함수 만들기

첫 번째 키워드가 항상 가장 중요한 키워드라는 보장은 없습니다.

이번에는 `keywords` 데이터를 조인하고 집계하는 흐름을 보기 위해 단순화된 대표 키워드로 사용합니다.

In [ ]:
def extract_first_keyword(keyword_text):
    if pd.isna(keyword_text):
        return np.nan

    try:
        parsed = ast.literal_eval(keyword_text)
    except (ValueError, SyntaxError):
        return np.nan

    if not isinstance(parsed, list) or len(parsed) == 0:
        return np.nan

    first_item = parsed[0]
    if not isinstance(first_item, dict):
        return np.nan

    return first_item.get("name", np.nan)

함수가 의도대로 동작하는지 원본 몇 개에 적용해 봅니다.

In [ ]:
keywords_raw["keywords"].head(5).apply(extract_first_keyword)

## 6. keywords_clean 만들기

원본 `keywords.csv`에서 이번 실습에 필요한 정보만 남겨 `keywords_clean`을 만듭니다.

지난 실습에서 `movies_metadata`를 정제했던 것처럼, 여기서도 같은 흐름으로 진행합니다.

- 필요한 컬럼만 선택
- 조인 키 이름을 `movie_id`로 표준화
- 조인 키 자료형 변환
- 대표 키워드 `main_keyword` 생성
- 결측치와 중복 확인
- 영화 1개당 1행이 되도록 정리

In [ ]:
keywords_clean = keywords_raw[["id", "keywords"]].copy()
keywords_clean = keywords_clean.rename(columns={"id": "movie_id"})

keywords_clean.head(3)

`keywords_clean`은 이후 `movies_clean`과 조인할 중간 테이블입니다.

따라서 한 영화가 여러 번 반복되지 않도록 `movie_id` 기준 중복을 확인하고 정리합니다. 이 과정은 지난 실습의 중복 제거와 같은 품질 점검 단계입니다.

In [ ]:
keywords_clean["movie_id"] = pd.to_numeric(keywords_clean["movie_id"], errors="coerce")
keywords_clean["main_keyword"] = keywords_clean["keywords"].apply(extract_first_keyword)

keywords_clean = keywords_clean[["movie_id", "main_keyword"]].copy()
keywords_clean = keywords_clean.dropna(subset=["movie_id"])
keywords_clean["movie_id"] = keywords_clean["movie_id"].astype("int64")

print("중복 제거 전 keywords_clean row 수:", len(keywords_clean))
print("중복 제거 전 movie_id 중복 수:", keywords_clean["movie_id"].duplicated().sum())

keywords_clean = keywords_clean.drop_duplicates(subset=["movie_id"], keep="first").copy()

print("중복 제거 후 keywords_clean row 수:", len(keywords_clean))
print("중복 제거 후 movie_id 중복 수:", keywords_clean["movie_id"].duplicated().sum())

keywords_clean.head(10)

In [ ]:
keywords_clean.isnull().sum()

## 7. 조인 전 점검

조인하기 전에 양쪽 테이블의 키 컬럼을 확인합니다.

확인할 내용은 단순합니다.

- `movie_id` 자료형이 같은가
- 기준 키에 중복이 남아 있지 않은가
- 조인 후에도 영화 1개당 1행 구조가 유지될 수 있는가

In [ ]:
movies_clean["movie_id"] = pd.to_numeric(movies_clean["movie_id"], errors="coerce")
movies_clean = movies_clean.dropna(subset=["movie_id"]).copy()
movies_clean["movie_id"] = movies_clean["movie_id"].astype("int64")

In [ ]:
print("movies_clean movie_id dtype:", movies_clean["movie_id"].dtype)
print("keywords_clean movie_id dtype:", keywords_clean["movie_id"].dtype)
print("movies_clean row 수:", len(movies_clean))
print("keywords_clean row 수:", len(keywords_clean))
print("movies_clean movie_id 중복 수:", movies_clean["movie_id"].duplicated().sum())
print("keywords_clean movie_id 중복 수:", keywords_clean["movie_id"].duplicated().sum())

## 8. movies_clean + keywords_clean 조인

`movies_clean`과 `keywords_clean`을 `movie_id` 기준으로 조인합니다.

이번 데이터에서는 두 테이블이 대부분 같은 영화 ID를 공유하므로 `inner join`을 사용합니다.

`inner join`은 두 테이블에 모두 존재하는 `movie_id`만 남기는 조인입니다. `main_keyword`가 비어 있는지는 조인 조건이 아니므로, `movie_id`가 매칭되면 `main_keyword`가 `NaN`이어도 결과에 남습니다.

키워드가 실제로 있는 영화만 분석할 때는 이후 분석 단계에서 `main_keyword` 결측을 제거합니다.

In [ ]:
# TODO: movies_clean과 keywords_clean을 movie_id 기준으로 inner join한다.
# 결과물 이름은 movies_with_keyword로 둔다.
# 힌트: pd.merge() 또는 DataFrame.merge()를 사용한다.
movies_with_keyword = None

print("movies_with_keyword shape:", movies_with_keyword.shape)
movies_with_keyword.head(5)

In [ ]:
print("조인 전 movies_clean row 수:", len(movies_clean))
print("조인 후 movies_with_keyword row 수:", len(movies_with_keyword))
print("조인 후 movie_id 중복 수:", movies_with_keyword["movie_id"].duplicated().sum())
print("main_keyword 결측 수:", movies_with_keyword["main_keyword"].isna().sum())

In [ ]:
print("movies_clean row 수:", len(movies_clean))
print("inner join 후 movies_with_keyword row 수:", len(movies_with_keyword))
print("조인 후 movie_id 중복 수:", movies_with_keyword["movie_id"].duplicated().sum())
print("main_keyword 결측 수:", movies_with_keyword["main_keyword"].isna().sum())

## 9. 비즈니스 질문을 위한 파생 컬럼 만들기

대표 키워드별 수익성과 효율을 보려면 `profit`, `roi`가 필요합니다.

- `profit`: 수익 - 예산
- `roi`: 수익 / 예산

`budget`이 0이면 `roi`를 계산하기 어렵기 때문에 `NaN`으로 둡니다.

In [ ]:
movies_with_keyword["profit"] = movies_with_keyword["revenue"] - movies_with_keyword["budget"]

In [ ]:
movies_with_keyword["roi"] = np.where(
    movies_with_keyword["budget"].notna() & (movies_with_keyword["budget"] > 0),
    movies_with_keyword["revenue"] / movies_with_keyword["budget"],
    np.nan,
)

연도별 pivot이 너무 듬성듬성해지는 문제를 줄이기 위해 `release_decade`도 만듭니다.

In [ ]:
movies_with_keyword["release_decade"] = (movies_with_keyword["release_year"] // 10) * 10
movies_with_keyword["release_decade"] = movies_with_keyword["release_decade"].astype("Int64")

In [ ]:
movies_with_keyword[[
    "title", "release_year", "release_decade", "budget", "revenue", "profit", "roi", "main_keyword"
]].head(5)

## 10. 중간 테이블 저장

`movies_with_keyword`는 아직 최종 집계표가 아닙니다.

한 행이 한 영화를 의미하고, 여러 분석에 재사용할 수 있으므로 중간 테이블로 저장합니다.

In [ ]:
movies_with_keyword.to_csv(MOVIES_WITH_KEYWORD_PATH, index=False)
MOVIES_WITH_KEYWORD_PATH

## 11. 질문 1: 신뢰할 만한 고평점 영화는 어느 시기에 많았을까?

평점 평균(`vote_average`)만 보면 평가 수가 너무 적은 영화가 과하게 좋아 보일 수 있습니다.

그래서 이번에는 다음 조건을 함께 사용합니다.

- `vote_average >= 7.0`
- `vote_count >= 100`

이렇게 하면 “평점도 높고, 어느 정도 평가 수도 있는 영화”를 볼 수 있습니다.

In [ ]:
# TODO: vote_average >= 7.0 이고 vote_count >= 100인 영화만 필터링한다.
# 결과물 이름은 reliable_high_rating_movies로 둔다.
# 힌트: 조건 2개를 & 로 묶고, 마지막에 .copy()를 붙인다.
reliable_high_rating_movies = None

print("신뢰 기준을 통과한 고평점 영화 수:", len(reliable_high_rating_movies))
reliable_high_rating_movies[["title", "release_year", "vote_average", "vote_count"]].head(10)

In [ ]:
# TODO: reliable_high_rating_movies를 release_year 기준으로 집계한다.
# 결과물 이름은 high_rating_by_year로 둔다.
# 결과 컬럼은 아래 이름을 사용한다.
# - high_rating_movie_count: 고평점 영화 수
# - avg_rating: 평균 평점
# - avg_vote_count: 평균 투표 수
# 힌트: groupby("release_year") + agg(...) + reset_index() + sort_values(...)
high_rating_by_year = None

high_rating_by_year.head(10)

## 12. 질문 2: 영화가 많이 나온 시기와 평균 평점이 높은 시기는 같을까?

이번에는 연도별로 영화 수, 평균 평점, 평균 투표 수, 평균 수익을 함께 봅니다.

`pd.options.display.float_format`을 설정했기 때문에 큰 숫자도 과학적 표기법 대신 쉼표가 들어간 숫자로 보입니다.

In [ ]:
# TODO: movies_with_keyword를 release_year 기준으로 집계한다.
# 결과물 이름은 year_summary로 둔다.
# 결과 컬럼은 아래 이름을 사용한다.
# - movie_count: 영화 수
# - avg_rating: 평균 평점
# - avg_vote_count: 평균 투표 수
# - avg_revenue: 평균 수익
# 힌트: agg() 안에서 새 컬럼명=(원본 컬럼명, 집계 함수) 형태를 사용한다.
year_summary = None

year_summary.head(10)

In [ ]:
year_summary.sort_values("movie_count", ascending=False).head(10)

In [ ]:
year_summary.sort_values("avg_rating", ascending=False).head(10)

## 13. 질문 3: 시기별로 어떤 대표 키워드가 많이 등장했을까?

처음에는 `연도 x 키워드` pivot을 생각할 수 있습니다.

하지만 실제로 해보면 연도와 키워드 조합이 너무 잘게 쪼개져 빈칸이 많아질 수 있습니다. 그래서 이번 실습에서는 `연대 x 대표 키워드`로 바꿔서 봅니다.

In [ ]:
movies_keyword_only = movies_with_keyword.dropna(subset=["main_keyword", "release_decade"]).copy()

top_keywords = (
    movies_keyword_only["main_keyword"]
    .value_counts()
    .head(8)
    .index
    .tolist()
)

top_keywords

In [ ]:
movies_keyword_top = movies_keyword_only[
    movies_keyword_only["main_keyword"].isin(top_keywords)
].copy()

movies_keyword_top[["title", "release_decade", "main_keyword"]].head(10)

In [ ]:
# TODO: release_decade를 행, main_keyword를 열로 두는 pivot table을 만든다.
# 결과물 이름은 decade_keyword_pivot으로 둔다.
# 값은 title 개수를 세어 채운다.
# 빈 조합은 0으로 채운다.
# 힌트: index, columns, values, aggfunc, fill_value를 지정한다.
decade_keyword_pivot = None

decade_keyword_pivot

## 14. 질문 4: 대표 키워드별 수익성과 효율은 어떻게 다를까?

이제 대표 키워드별로 여러 지표를 함께 계산합니다.

이번 결과는 특정 질문에 답하기 위해 만들어진 비즈니스 테이블에 가깝습니다.

In [ ]:
# TODO: movies_keyword_top을 main_keyword 기준으로 집계한다.
# 결과물 이름은 keyword_business로 둔다.
# 결과 컬럼은 아래 이름을 사용한다.
# - movie_count: 영화 수
# - avg_rating: 평균 평점
# - avg_vote_count: 평균 투표 수
# - avg_revenue: 평균 수익
# - avg_profit: 평균 이익
# - avg_roi: 평균 ROI
# 힌트: groupby("main_keyword") + agg(...) + reset_index() + sort_values(...)
keyword_business = None

keyword_business

## 15. wide table과 long table

`keyword_business`는 사람이 표로 읽기 좋은 wide table입니다.

하지만 여러 지표를 같은 방식으로 시각화하거나 추가 가공하려면 `metric`, `value` 구조의 long table이 더 편할 때가 많습니다.

In [ ]:
# TODO: keyword_business를 wide format에서 long format으로 변환한다.
# 결과물 이름은 keyword_business_long으로 둔다.
# 고정할 컬럼(id_vars)은 main_keyword다.
# 세로로 내릴 지표 컬럼(value_vars)은 아래 6개다.
# - movie_count
# - avg_rating
# - avg_vote_count
# - avg_revenue
# - avg_profit
# - avg_roi
# 결과 컬럼명은 metric, value를 사용한다.
keyword_business_long = None

keyword_business_long.head(12)

## 16. long table을 활용한 시각화

아래 시각화 코드는 직접 작성하는 대상이라기보다, `melt()` 이후 long format이 왜 시각화에 편한지 보여주기 위한 예시입니다.

지표마다 단위와 스케일이 다르므로 `sharey=False`로 나누어 봅니다.

In [ ]:
g = sns.catplot(
    data=keyword_business_long,
    x="main_keyword",
    y="value",
    col="metric",
    kind="bar",
    col_wrap=3,
    sharey=False,
    height=4,
    aspect=1.2,
)

g.set_xticklabels(rotation=45)
g.fig.subplots_adjust(top=0.88)
g.fig.suptitle("대표 키워드별 지표 비교")
plt.show()

## 17. 결과 저장

중간 테이블과 비즈니스 테이블을 저장합니다.

- `movies_with_keyword`: 조인과 파생 컬럼이 반영된 중간 테이블
- `year_business_table`: 연도별 요약 비즈니스 테이블
- `decade_keyword_pivot`: 연대 x 대표 키워드 pivot table
- `keyword_business_table_wide`: 대표 키워드별 wide table
- `keyword_business_table_long`: 시각화/후속 가공용 long table

In [ ]:
year_summary.to_csv(YEAR_BUSINESS_PATH, index=False)
decade_keyword_pivot.to_csv(DECADE_KEYWORD_PIVOT_PATH)
keyword_business.to_csv(KEYWORD_BUSINESS_WIDE_PATH, index=False)
keyword_business_long.to_csv(KEYWORD_BUSINESS_LONG_PATH, index=False)

print("저장 완료:")
print("-", MOVIES_WITH_KEYWORD_PATH)
print("-", YEAR_BUSINESS_PATH)
print("-", DECADE_KEYWORD_PIVOT_PATH)
print("-", KEYWORD_BUSINESS_WIDE_PATH)
print("-", KEYWORD_BUSINESS_LONG_PATH)

## 정리

이번 실습에서는 다음 흐름을 확인했습니다.

```text
raw keywords.csv
→ keywords_clean
→ movies_clean과 inner join
→ movies_with_keyword 중간 테이블
→ groupby / pivot / melt 기반 business table
```

핵심은 `join`, `groupby`, `pivot_table`, `melt`를 각각 따로 외우는 것이 아니라, Transform 단계에서 테이블의 목적이 어떻게 달라지는지 이해하는 것입니다.